In [1]:
import os
import glob
import yaml

import numpy as np
import pandas as pd

from math import ceil

from matplotlib import cm

import tifffile

import zarr
import napari
import dask.array as da

from utils.utility_functions import single_channel_pyramid

In [2]:
# specify which clustering(s) in main.csv to visualize
clusterings = ['VAE30', 'VAE9_VIG7', 'Seg']

In [3]:
# I/O

# read single-cell data
main = pd.read_csv(os.path.join(os.getcwd(), 'input/main.csv'))

# read OME-TIFF, segmentation outlines, and H&E channels
tif_path = os.path.join(os.getcwd(), 'input/CyCIF-1A_image.ome.tif')
seg_path = os.path.join(os.getcwd(), 'input/CyCIF-1A_seg_outlines.ome.tif')
he_path = os.path.join(os.getcwd(), 'input/CyCIF-1A_hema_eosin.ome.tif')

# import markers.csv
markers = pd.read_csv(os.path.join(os.getcwd(), 'input/CyCIF-1A_mcmicro_markers.csv'))

# import image contrast settings
with open(os.path.join(os.getcwd(), 'input/CyCIF-1A_cylinter_contrast_limits.yml')) as f:
    contrast_limits = yaml.safe_load(f)

# the parquet file at the path below is being read because "main.csv" 
# uses trimmed marker channel names as column headers that differ from the raw channel names used 
# in the markers.csv file, which is itself used to index channels in the OME-TIFF image.
for_channels = pd.read_parquet(
    os.path.join(os.getcwd(), 'input/CyCIF-1A_clean_cylinter_clustering_3d_leiden.parquet')
)

# isolate antibodies of interest
abx_channels = [i for i in for_channels.columns if 'nucleiRingMask' in i if 'Hoechst' not in i]

In [4]:
# add H&E image to Napari viewer as separate RGB channels
for color, channel in zip(['red', 'green', 'blue'], [0, 1, 2]):

    img, min, max = single_channel_pyramid(glob.glob(he_path)[0], channel=channel)

    if channel == 0:
        viewer = napari.view_image(
            img, rgb=False, colormap=color, blending='additive',
            visible=False, name=f'H&E_{color}', contrast_limits=(min, max)
        )
    else:
        viewer.add_image(
            img, rgb=False, colormap=color, blending='additive',
            visible=False, name=f'H&E_{color}', contrast_limits=(min, max)
        )

In [5]:
# OPTIONAL: add H&E image to Napari viewer as a single channel image

# from lazy_ops import DatasetView
# tiff = tifffile.TiffFile(he_path, is_ome=False)
# pyramid = [
#     zarr.open(tiff.series[0].levels[0].aszarr())[i] for i in
#     list(range(len(tiff.series[0].levels)))
#     ]
# pyramid = [DatasetView(i).lazy_transpose([1, 2, 0]) for i in pyramid]
# pyramid = [da.from_zarr(z) for z in pyramid]
#
# viewer = napari.view_image(pyramid, rgb=True, name='H&E')

In [6]:
# add DNA1 channel to image viewer
dna, min, max = single_channel_pyramid(glob.glob(tif_path)[0], channel=0)
viewer.add_image(
    dna, rgb=False, blending='additive',
    colormap='gray', visible=True, opacity=0.8,
    name='DNA1', contrast_limits=(min, max)
)

<Image layer 'DNA1' at 0x143ff83a0>

In [7]:
# add marker channels to image viewer and apply previously defined contrast limits
for ch in abx_channels:
    ch = ch.rsplit('_', 1)[0]
    channel_number = markers['channel_number'][markers['marker_name'] == ch]
    
    img, min, max = single_channel_pyramid(
        glob.glob(tif_path)[0], channel=(channel_number.item() - 1)
    )
    viewer.add_image(
        img, rgb=False, blending='additive', colormap='green', visible=False, name=ch,
        contrast_limits=(min, max)
    )
for ch in abx_channels:
    ch = ch.rsplit('_', 1)[0]
    viewer.layers[ch].contrast_limits = (
        contrast_limits[ch][0], contrast_limits[ch][1])

In [8]:
# box size
n = 14
half_n = n / 2

# add centroids of cells in each clustering
for clustering in clusterings:
    
    num_colors = len(list(cm.tab20.colors))
    num_clusters = len(main[clustering].unique())
    palette_multiplier = ceil(num_clusters / num_colors)
    colors = list(cm.tab20.colors) * palette_multiplier
    colors = colors[0:num_clusters]
    colors.reverse()

    if clustering == 'VAE9':
        
        # 9um (14px) meta-cluster groups
        epithelial = [24, 25, 20, 14, 31, 9, 10, 3, 5, 6, 11, 15, 19, 27, 28] 
        immune = [26, 30, 8, 12, 16, 21, 7, 2, 22]
        stromal = [1, 4, 13, 18, 29, 0, 17, 23]
        cluster_order = epithelial + immune + stromal
        cluster_order.reverse()
        
        my_dict = dict(zip(sorted(main[clustering].unique(), reverse=True), colors))
        sorted_dict = {key: my_dict[key] for key in cluster_order if key in my_dict}

        for cluster, c in sorted_dict.items():
            centroids = main[['Y_centroid', 'X_centroid']][main[clustering] == cluster]
            viewer.add_points(
                centroids, name=f'{clustering}_{cluster}', face_color=np.array(c), border_color='white',
                border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
            )

    else:
        for c, cluster in zip(colors, sorted(main[clustering].unique(), reverse=True)):
            centroids = main[['Y_centroid', 'X_centroid']][main[clustering] == cluster]
            viewer.add_points(
                centroids, name=f'{clustering}_{cluster}', face_color=np.array(c), border_color='white',
                border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
            )
        
# centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 13) & (main['VAE9_VIG7'] == 15)]
# viewer.add_points(
#     centroids, name='Seg13_V15', face_color='#c7c7c7', border_color='white',
#     border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 2)]
# viewer.add_points(
#     centroids, name='Seg3_V2', face_color='#ff7f0e', border_color='white',
#     border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
# )
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[1, 1, 1, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='Seg3_V2_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 7)]
# viewer.add_points(
#     centroids, name='Seg3_7', face_color='#ff9896', border_color='white',
#     border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
# )
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[1, 1, 1, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='Seg3_7_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 12)]
# viewer.add_points(
#     centroids, name='S3_V12', face_color='#e377c2', border_color='white',
#     border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
# )
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[1, 1, 1, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='S3_V12_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 14)]
# viewer.add_points(
#     centroids, name='S3_V14', face_color='#55aaff', border_color='white',
#     border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
# )
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[1, 1, 1, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='S3_V14_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 16)]
# viewer.add_points(
#     centroids, name='Seg3_V16', face_color='#bcbd22', border_color='white',
#     border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
# )
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[1, 1, 1, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='S3_V16_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['VAE9_VIG7'] == 23)]
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[1, 1, 0, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='V23_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['VAE9_VIG7'] == 0)]
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[0, 1, 0, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='V0_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['VAE9_VIG7'] == 3)]
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[0, 1, 0, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='V3_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['VAE9_VIG7'] == 5)]
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[0, 1, 0, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='V5_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['VAE9_VIG7'] == 21)]
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[0, 1, 0, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='V21_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['VAE9_VIG7'] == 22)]
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[0, 1, 0, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='V22_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['VAE9_VIG7'] == 10)]
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[0, 1, 0, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='V10_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['VAE9_VIG7'] == 1)]
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[0, 1, 0, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='V1_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['VAE9_VIG7'] == 6)]
# boxes = []
# for point in np.array(centroids):
#     row, col = point
#     box = [
#         [row - half_n, col - half_n],  # top-left
#         [row - half_n, col + half_n],  # top-right
#         [row + half_n, col + half_n],  # bottom-right
#         [row + half_n, col - half_n],  # bottom-left
#     ]
#     boxes.append(box)
# viewer.add_shapes(
#     np.array(boxes),
#     shape_type='polygon',
#     edge_color=[0, 1, 0, 1],
#     face_color='transparent',
#     edge_width=1,
#     visible=False,
#     name='V6_box'
# )

# centroids = main[['Y_centroid', 'X_centroid']][main['VAE9_VIG7'].isin([4])]
# # [4, 9, 11, 13, 14, 15, 18, 20]
# point_properties = {'probability': main['FOXP3_570'][main['VAE9_VIG7'].isin([4])]}

# viewer.add_points(
#     centroids, name='FOXP3', properties=point_properties,
#     face_color='probability', face_colormap='viridis',
#     border_width=0.0, size=10.0, opacity=1.0, blending='opaque',
#     visible=False
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 13) & (main['VAE9_VIG7'] == 4)]
# viewer.add_points(
#     centroids, name='test13_4', face_color='#1f77b4', border_color='white',
#     border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
# )

# centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 15) & (main['VAE9_VIG7'] == 4)]
# viewer.add_points(
#     centroids, name='test15_4', face_color='#ff7f0e', border_color='white',
#     border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
# )

In [9]:
# add segmentation outlines to image viewer
seg, min, max = single_channel_pyramid(glob.glob(seg_path)[0], channel=0)
viewer.add_image(
    seg, rgb=False, blending='additive',
    colormap='gray', visible=False,
    name='segmentation', opacity=0.3, contrast_limits=(min, max)
)

<Image layer 'segmentation' at 0x16b9610d0>

In [10]:
# run image viewer
viewer.scale_bar.visible = True
viewer.scale_bar.unit = 'um'